# Pipeline de dados com PySpark: desigualdade no ENEM 2023

**Arquitetura medalhão (bronze → prata → ouro) com os microdados oficiais do INEP**

Este notebook é um tutorial completo e reproduzível de engenharia de dados. Ele baixa os microdados do ENEM 2023 (cerca de 4 milhões de inscritos e 1,7 GB de CSV), processa tudo com **Apache Spark** e responde a uma pergunta concreta:

> **Quanto a renda da família e o tipo de escola pesam na nota do ENEM, no Brasil, em São Paulo e no Vale do Paraíba?**

**O que você vai aprender**

- Ler um CSV grande e "problemático" (latin-1, separador `;`) com Spark
- Montar as três camadas da arquitetura medalhão e entender o papel de cada uma
- Converter CSV para Parquet e medir o ganho de espaço
- Traduzir códigos com `create_map` e *broadcast join*, sem UDFs lentas
- Particionar dados e ver o Spark ler só a partição necessária (`explain`)
- Criar checagens de qualidade de dados
- Gerar tabelas analíticas e gráficos a partir da camada ouro

**Como rodar:** no menu, clique em **Ambiente de execução → Executar tudo**. Leva de 15 a 25 minutos. Não é preciso instalar nada no seu computador. Antes, baixe o `microdados_enem_2023.zip` no [site do INEP](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem) e envie para o **Meu Drive** (veja a seção 2).

```text
 CSV do INEP (1,7 GB, latin-1, separador ;)
   │
   ▼  BRONZE  cópia fiel em Parquet, tudo como texto, com metadados de ingestão
   │
   ▼  PRATA   só as colunas úteis, tipos corretos, códigos traduzidos, regras de qualidade
   │
   ▼  OURO    tabelas agregadas, prontas para gráficos e dashboards
```

*Projeto de Extensão Universitária, Ciência da Computação (UNIP), por Pedro Henrique Gomes Figueiredo.*

## 1. Preparar o ambiente

O Google Colab já vem com Java, que o Spark exige. Esta célula instala o PySpark (se ainda não estiver instalado) e cria as pastas de cada camada.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"], check=True)

import os, time, zipfile, shutil
from pathlib import Path

BASE = Path(os.environ.get("ENEM_BASE", "/content/enem"))
PASTA_RAW = BASE / "raw"
PASTA_BRONZE = BASE / "bronze" / "enem_2023"
PASTA_PRATA = BASE / "prata" / "enem_2023"
PASTA_OURO = BASE / "ouro"
PASTA_SAIDA = BASE / "saida"
PASTA_GRAFICOS = PASTA_SAIDA / "graficos"
for pasta in (PASTA_RAW, PASTA_OURO, PASTA_GRAFICOS):
    pasta.mkdir(parents=True, exist_ok=True)

URL_INEP = "https://download.inep.gov.br/microdados/microdados_enem_2023.zip"
ARQUIVO_ZIP = PASTA_RAW / "microdados_enem_2023.zip"
ARQUIVO_CSV = PASTA_RAW / "MICRODADOS_ENEM_2023.csv"

def tamanho(caminho):
    """Tamanho em bytes de um arquivo ou de uma pasta inteira."""
    caminho = Path(caminho)
    if caminho.is_file():
        return caminho.stat().st_size
    return sum(f.stat().st_size for f in caminho.rglob("*") if f.is_file())

print("Pastas prontas em", BASE)

## 2. Obter os microdados do INEP

O arquivo compactado tem cerca de 600 MB. Esta célula procura o zip em dois lugares, nesta ordem:

1. **No seu Google Drive:** se você já baixou o `microdados_enem_2023.zip` no site do INEP e enviou para o **Meu Drive**, o Colab pede permissão para acessar o Drive e copia o arquivo de lá. É o caminho mais confiável.
2. **Direto do site do INEP:** se não achar no Drive, tenta baixar. Às vezes o site do INEP recusa conexões do Colab; nesse caso, use o caminho 1.

Depois extraímos só o CSV de participantes, porque o zip também traz PDFs, dicionários e scripts.

In [ ]:
def procurar_no_drive():
    """Monta o Google Drive e procura o zip do ENEM 2023 no Meu Drive (e uma pasta abaixo)."""
    try:
        from google.colab import drive
    except ImportError:
        return None
    drive.mount("/content/drive")
    raiz = Path("/content/drive/MyDrive")
    achados = list(raiz.glob("microdados_enem_2023*.zip")) + list(raiz.glob("*/microdados_enem_2023*.zip"))
    return achados[0] if achados else None

def baixar(url, destino):
    """Baixa com wget, mostrando o progresso e retomando downloads interrompidos."""
    resultado = subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", str(destino), url])
    return resultado.returncode == 0

if ARQUIVO_CSV.exists():
    print("CSV já extraído; pulando esta etapa.")
else:
    if not ARQUIVO_ZIP.exists() or tamanho(ARQUIVO_ZIP) < 100_000_000:
        no_drive = procurar_no_drive()
        if no_drive:
            print("Copiando do Google Drive:", no_drive)
            shutil.copy(no_drive, ARQUIVO_ZIP)
        else:
            print("Zip não encontrado no Drive. Tentando baixar de", URL_INEP)
            if not baixar(URL_INEP, ARQUIVO_ZIP):
                raise RuntimeError(
                    "O download do INEP falhou. Baixe o microdados_enem_2023.zip na página do INEP, "
                    "envie para o Meu Drive (drive.google.com) e rode esta célula de novo.")
    with zipfile.ZipFile(ARQUIVO_ZIP) as z:
        membro = next(m for m in z.namelist() if m.upper().endswith("MICRODADOS_ENEM_2023.CSV"))
        print("Extraindo", membro)
        with z.open(membro) as origem, open(ARQUIVO_CSV, "wb") as destino:
            shutil.copyfileobj(origem, destino, length=16 * 1024 * 1024)
    ARQUIVO_ZIP.unlink()  # libera espaço no disco do Colab (o original continua no Drive)

print(f"CSV pronto: {tamanho(ARQUIVO_CSV) / 1e9:.2f} GB")

## 3. Iniciar o Spark

No Colab o Spark roda em modo `local[*]`: um único computador, usando todos os núcleos como se fossem "executores" de um cluster. O código é o mesmo que rodaria num cluster de verdade (Databricks, EMR, Dataproc).

Um conceito importante antes de começar: o Spark é **preguiçoso** (*lazy evaluation*). Transformações como `select`, `filter` e `join` só montam um plano. O trabalho pesado acontece quando uma **ação** pede um resultado: `count`, `write`, `show`, `toPandas`.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("enem-2023-medalhao")
         .master("local[*]")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "16")  # o padrão (200) é exagerado para 1 máquina
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| núcleos em uso:", spark.sparkContext.defaultParallelism)

## 4. Camada bronze: guardar o dado bruto, do jeito que veio

A bronze é o "arquivo morto" confiável do pipeline. Regras:

- **Nada é corrigido aqui.** Todas as colunas são lidas como texto (sem `inferSchema`), exatamente como o INEP publicou.
- Acrescentamos **metadados de ingestão** (arquivo de origem e data da carga) para rastreabilidade.
- Gravamos em **Parquet**, um formato colunar e comprimido. Ler uma coluna não exige ler o arquivo inteiro.

Se uma regra de limpeza estiver errada no futuro, refazemos a prata a partir da bronze, sem baixar tudo de novo.

In [ ]:
inicio = time.time()
bronze_bruto = (spark.read
                .option("header", True)
                .option("sep", ";")
                .option("encoding", "ISO-8859-1")   # o INEP publica em latin-1
                .csv(str(ARQUIVO_CSV))
                .withColumn("_arquivo_origem", F.lit(ARQUIVO_CSV.name))
                .withColumn("_ingerido_em", F.current_timestamp()))

bronze_bruto.write.mode("overwrite").parquet(str(PASTA_BRONZE))
print(f"Bronze gravada em {time.time() - inicio:.0f} s")

bronze = spark.read.parquet(str(PASTA_BRONZE))
print(f"Linhas:  {bronze.count():,}".replace(",", "."))
print(f"Colunas: {len(bronze.columns)}")
print(f"CSV original: {tamanho(ARQUIVO_CSV) / 1e9:.2f} GB  ->  Parquet: {tamanho(PASTA_BRONZE) / 1e9:.2f} GB")

## 5. Camada prata: dado limpo, tipado e com significado

Na prata o dado passa a ser confiável para análise:

1. **Seleção** de 20 das mais de 70 colunas, renomeadas em português e em minúsculas
2. **Tipos corretos**: notas viram `double`, códigos de presença viram `int`
3. **Tradução de códigos** com `create_map` (tipo de escola, cor/raça) e com um *broadcast join* (faixas de renda e municípios do Vale do Paraíba). As duas técnicas rodam dentro do Spark, sem as UDFs em Python, que são lentas
4. **Regras de negócio**: quem esteve presente nas quatro provas e a média geral das cinco notas
5. **Checagens de qualidade** antes de gravar
6. **Particionamento por UF** para acelerar consultas regionais

In [ ]:
# ---- dicionários (códigos do dicionário de dados do ENEM 2023) ----
TIPO_ESCOLA = {"1": "Não respondeu", "2": "Pública", "3": "Privada"}
COR_RACA = {"0": "Não declarado", "1": "Branca", "2": "Preta", "3": "Parda",
            "4": "Amarela", "5": "Indígena", "6": "Não informado"}

# Q006: renda mensal da família, em salários mínimos (SM de 2023 = R$ 1.320,00)
RENDA = [("A", "Nenhuma renda", "Até 1 SM"), ("B", "Até 1 SM", "Até 1 SM"),
         ("C", "1 a 1,5 SM", "1 a 2 SM"), ("D", "1,5 a 2 SM", "1 a 2 SM"),
         ("E", "2 a 2,5 SM", "2 a 3 SM"), ("F", "2,5 a 3 SM", "2 a 3 SM"),
         ("G", "3 a 4 SM", "3 a 5 SM"), ("H", "4 a 5 SM", "3 a 5 SM"),
         ("I", "5 a 6 SM", "5 a 10 SM"), ("J", "6 a 7 SM", "5 a 10 SM"),
         ("K", "7 a 8 SM", "5 a 10 SM"), ("L", "8 a 9 SM", "5 a 10 SM"),
         ("M", "9 a 10 SM", "5 a 10 SM"), ("N", "10 a 12 SM", "Mais de 10 SM"),
         ("O", "12 a 15 SM", "Mais de 10 SM"), ("P", "15 a 20 SM", "Mais de 10 SM"),
         ("Q", "Mais de 20 SM", "Mais de 10 SM")]
GRUPOS_RENDA = ["Até 1 SM", "1 a 2 SM", "2 a 3 SM", "3 a 5 SM", "5 a 10 SM", "Mais de 10 SM"]

# 39 municípios da Região Metropolitana do Vale do Paraíba e Litoral Norte (código IBGE)
VALE_DO_PARAIBA = {
    "3508504": "Caçapava", "3520202": "Igaratá", "3524402": "Jacareí", "3524907": "Jambeiro",
    "3531704": "Monteiro Lobato", "3535606": "Paraibuna", "3546009": "Santa Branca",
    "3549904": "São José dos Campos", "3509700": "Campos do Jordão", "3526308": "Lagoinha",
    "3532306": "Natividade da Serra", "3538006": "Pindamonhangaba", "3542305": "Redenção da Serra",
    "3548203": "Santo Antônio do Pinhal", "3548609": "São Bento do Sapucaí",
    "3550001": "São Luiz do Paraitinga", "3554102": "Taubaté", "3554805": "Tremembé",
    "3502507": "Aparecida", "3508603": "Cachoeira Paulista", "3509957": "Canas", "3513603": "Cunha",
    "3518404": "Guaratinguetá", "3527207": "Lorena", "3538501": "Piquete", "3540754": "Potim",
    "3544301": "Roseira", "3503158": "Arapeí", "3503505": "Areias", "3504909": "Bananal",
    "3513405": "Cruzeiro", "3526605": "Lavrinhas", "3541901": "Queluz",
    "3549607": "São José do Barreiro", "3552007": "Silveiras", "3510500": "Caraguatatuba",
    "3520400": "Ilhabela", "3550704": "São Sebastião", "3555406": "Ubatuba",
}

def traduzir(dicionario, coluna):
    """Troca códigos por textos com um mapa nativo do Spark (sem UDF)."""
    mapa = F.create_map([F.lit(x) for par in dicionario.items() for x in par])
    return mapa[F.col(coluna)]

def nota(coluna):
    """Notas chegam como texto; trocamos vírgula por ponto, por segurança, e convertemos."""
    return F.regexp_replace(F.col(coluna), ",", ".").cast("double")

renda_df = spark.createDataFrame(
    [(cod, i, faixa, grupo, GRUPOS_RENDA.index(grupo)) for i, (cod, faixa, grupo) in enumerate(RENDA)],
    "q006 string, renda_ordem int, renda_faixa string, renda_grupo string, renda_grupo_ordem int")
vale_df = spark.createDataFrame(list(VALE_DO_PARAIBA.items()), "co_municipio_prova string, municipio_vale string")

COLUNAS_NECESSARIAS = ["NU_INSCRICAO", "TP_FAIXA_ETARIA", "TP_SEXO", "TP_COR_RACA", "TP_ESCOLA", "IN_TREINEIRO",
                       "CO_MUNICIPIO_PROVA", "NO_MUNICIPIO_PROVA", "SG_UF_PROVA", "TP_PRESENCA_CN", "TP_PRESENCA_CH",
                       "TP_PRESENCA_LC", "TP_PRESENCA_MT", "TP_STATUS_REDACAO", "NU_NOTA_CN", "NU_NOTA_CH",
                       "NU_NOTA_LC", "NU_NOTA_MT", "NU_NOTA_REDACAO", "Q006"]
faltando = [c for c in COLUNAS_NECESSARIAS if c not in bronze.columns]
assert not faltando, f"Colunas ausentes no arquivo do INEP: {faltando}"
print("Todas as", len(COLUNAS_NECESSARIAS), "colunas necessárias estão no arquivo.")

In [ ]:
prata_plano = (bronze
    .select(
        F.col("NU_INSCRICAO").alias("inscricao"),
        F.col("TP_FAIXA_ETARIA").cast("int").alias("faixa_etaria"),
        F.col("TP_SEXO").alias("sexo"),
        traduzir(COR_RACA, "TP_COR_RACA").alias("cor_raca"),
        traduzir(TIPO_ESCOLA, "TP_ESCOLA").alias("tipo_escola"),
        F.col("IN_TREINEIRO").cast("int").alias("treineiro"),
        F.col("CO_MUNICIPIO_PROVA").alias("co_municipio_prova"),
        F.col("NO_MUNICIPIO_PROVA").alias("municipio_prova"),
        F.col("SG_UF_PROVA").alias("uf_prova"),
        *[F.col(f"TP_PRESENCA_{a}").cast("int").alias(f"presenca_{a.lower()}") for a in ("CN", "CH", "LC", "MT")],
        F.col("TP_STATUS_REDACAO").cast("int").alias("status_redacao"),
        nota("NU_NOTA_CN").alias("nota_cn"),
        nota("NU_NOTA_CH").alias("nota_ch"),
        nota("NU_NOTA_LC").alias("nota_lc"),
        nota("NU_NOTA_MT").alias("nota_mt"),
        nota("NU_NOTA_REDACAO").alias("nota_redacao"),
        F.col("Q006").alias("q006"))
    # tabelas pequenas vão inteiras para cada núcleo (broadcast): o join não embaralha os 4 milhões de linhas
    .join(F.broadcast(renda_df), "q006", "left")
    .join(F.broadcast(vale_df), "co_municipio_prova", "left")
    .withColumn("vale_do_paraiba", F.col("municipio_vale").isNotNull())
    # presença: 0 = faltou, 1 = presente, 2 = eliminado
    .withColumn("presente_todas", (F.col("presenca_cn") == 1) & (F.col("presenca_ch") == 1)
                                  & (F.col("presenca_lc") == 1) & (F.col("presenca_mt") == 1))
    .withColumn("media_geral", F.when(F.col("presente_todas"),
        F.round((F.col("nota_cn") + F.col("nota_ch") + F.col("nota_lc") + F.col("nota_mt") + F.col("nota_redacao")) / 5, 2))))

prata_plano.printSchema()

### 5.1 Checagens de qualidade

Antes de gravar a prata, conferimos se o dado faz sentido. Num pipeline de produção, uma checagem que falha interrompe a carga e avisa o time.

In [ ]:
NOTAS = ["nota_cn", "nota_ch", "nota_lc", "nota_mt", "nota_redacao"]
qualidade = prata_plano.agg(
    F.count("*").alias("linhas"),
    F.countDistinct("inscricao").alias("inscricoes_unicas"),
    F.sum(F.col("presente_todas").cast("int")).alias("presentes_nas_4_provas"),
    F.sum(F.col("tipo_escola").isNull().cast("int")).alias("tipo_escola_sem_traducao"),
    F.sum((F.col("q006").isNotNull() & F.col("renda_faixa").isNull()).cast("int")).alias("renda_sem_traducao"),
    F.sum((F.greatest(*NOTAS) > 1000).cast("int")).alias("notas_acima_de_1000"),
    F.sum((F.least(*NOTAS) < 0).cast("int")).alias("notas_negativas"),
    F.sum((F.col("presente_todas") & F.col("media_geral").isNull()).cast("int")).alias("presentes_sem_nota"),
).first().asDict()

for chave, valor in qualidade.items():
    print(f"{chave:<28} {valor:>12,}".replace(",", "."))

# Regras: o que for crítico interrompe o pipeline; o resto vira alerta
assert qualidade["presentes_nas_4_provas"] > 0, "Nenhum participante presente: o arquivo parece errado"
alertas = []
if qualidade["linhas"] != qualidade["inscricoes_unicas"]:
    alertas.append("há inscrições repetidas")
if qualidade["notas_acima_de_1000"] or qualidade["notas_negativas"]:
    alertas.append("há notas fora da escala de 0 a 1000")
if qualidade["tipo_escola_sem_traducao"] or qualidade["renda_sem_traducao"]:
    alertas.append("há códigos sem tradução nos dicionários")
print("\n" + ("ALERTA: " + "; ".join(alertas) if alertas else "Todas as checagens passaram."))

In [ ]:
inicio = time.time()
prata_plano.write.mode("overwrite").partitionBy("uf_prova").parquet(str(PASTA_PRATA))
prata = spark.read.parquet(str(PASTA_PRATA))
print(f"Prata gravada em {time.time() - inicio:.0f} s: {tamanho(PASTA_PRATA) / 1e6:.0f} MB, uma pasta por UF")
print(sorted(p.name for p in PASTA_PRATA.iterdir() if p.is_dir())[:6], "...")

### 5.2 Particionamento na prática

Como a prata está particionada por UF, uma consulta que filtra São Paulo lê **só a pasta `uf_prova=SP`**. Procure por `PartitionFilters` no plano de execução abaixo: é o Spark avisando que vai pular as outras 26 pastas.

In [ ]:
prata.filter(F.col("uf_prova") == "SP").select("media_geral").explain()

## 6. Camada ouro: respostas prontas

A ouro tem tabelas pequenas e agregadas, feitas para responder perguntas e alimentar gráficos e dashboards.

**Recorte da análise:** inscritos presentes nas quatro provas, sem os treineiros (quem fez a prova só para testar). O **tipo de escola** só é informado por quem concluiu o ensino médio em 2023. Por isso as tabelas por escola consideram apenas esses concluintes.

Usamos `cache()` porque a mesma base é lida várias vezes. Assim o Spark guarda o resultado em memória depois da primeira leitura.

In [ ]:
analise = prata.filter(F.col("presente_todas") & (F.col("treineiro") == 0)).cache()

def tres_recortes(df):
    """Empilha o mesmo dado em três recortes: Brasil, São Paulo e Vale do Paraíba."""
    return (df.withColumn("recorte", F.lit("Brasil"))
            .unionByName(df.filter(F.col("uf_prova") == "SP").withColumn("recorte", F.lit("São Paulo")))
            .unionByName(df.filter(F.col("vale_do_paraiba")).withColumn("recorte", F.lit("Vale do Paraíba"))))

METRICAS = [F.count(F.lit(1)).alias("participantes"),
            F.round(F.avg("media_geral"), 1).alias("media_geral"),
            F.round(F.avg("nota_mt"), 1).alias("media_matematica"),
            F.round(F.avg("nota_redacao"), 1).alias("media_redacao")]

escolas = ["Pública", "Privada"]

ouro = {}
ouro["renda"] = (tres_recortes(analise)
    .groupBy("recorte", "renda_ordem", "renda_faixa").agg(*METRICAS)
    .orderBy("recorte", "renda_ordem"))

ouro["renda_e_escola"] = (tres_recortes(analise.filter(F.col("tipo_escola").isin(escolas)))
    .groupBy("recorte", "renda_grupo_ordem", "renda_grupo", "tipo_escola").agg(*METRICAS)
    .orderBy("recorte", "renda_grupo_ordem", "tipo_escola"))

ouro["escola"] = (tres_recortes(analise.filter(F.col("tipo_escola").isin(escolas)))
    .groupBy("recorte", "tipo_escola").agg(*METRICAS)
    .orderBy("recorte", "tipo_escola"))

ouro["municipios_vale"] = (analise
    .filter(F.col("vale_do_paraiba") & F.col("tipo_escola").isin(escolas))
    .groupBy("municipio_vale")
    .pivot("tipo_escola", escolas)
    .agg(F.count(F.lit(1)).alias("n"), F.round(F.avg("media_geral"), 1).alias("media"))  # Spark 4 não aceita count("*") em pivot
    .withColumnRenamed("municipio_vale", "municipio")
    .withColumnRenamed("Pública_n", "concluintes_publica").withColumnRenamed("Pública_media", "media_publica")
    .withColumnRenamed("Privada_n", "concluintes_privada").withColumnRenamed("Privada_media", "media_privada")
    .withColumn("diferenca", F.round(F.col("media_privada") - F.col("media_publica"), 1))
    .orderBy(F.col("concluintes_publica").desc_nulls_last()))

inscritos = prata.filter(F.col("treineiro") == 0)
ouro["presenca"] = (tres_recortes(inscritos).groupBy("recorte")
    .agg(F.count("*").alias("inscritos"), F.sum(F.col("presente_todas").cast("int")).alias("presentes"))
    .withColumn("abstencao", F.round(1 - F.col("presentes") / F.col("inscritos"), 4))
    .orderBy("recorte"))

for nome, df in ouro.items():
    df.write.mode("overwrite").parquet(str(PASTA_OURO / nome))
    tabela = df.toPandas()
    tabela.to_csv(PASTA_SAIDA / f"ouro_{nome}.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
    print(f"ouro/{nome:<16} {len(tabela):>4} linhas")

## 7. Resultados

As frases abaixo são montadas a partir das tabelas da camada ouro. Os números vêm dos dados, não foram digitados à mão.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

presenca = ouro["presenca"].toPandas().set_index("recorte")
escola = ouro["escola"].toPandas()
renda = ouro["renda"].toPandas()
municipios = ouro["municipios_vale"].toPandas()

fmt = lambda x: f"{x:,.1f}".replace(",", "X").replace(".", ",").replace("X", ".")
for recorte in ["Brasil", "São Paulo", "Vale do Paraíba"]:
    e = escola[escola.recorte == recorte].set_index("tipo_escola")
    r = renda[renda.recorte == recorte].dropna(subset=["renda_faixa"])
    pobre, rico = r.iloc[0], r.iloc[-1]
    print(f"\n{recorte.upper()}")
    print(f"  Abstenção (faltou a pelo menos uma prova): {presenca.loc[recorte, 'abstencao']:.1%}".replace(".", ","))
    if {"Pública", "Privada"} <= set(e.index):
        dif = e.loc["Privada", "media_geral"] - e.loc["Pública", "media_geral"]
        print(f"  Concluintes de escola privada: média {fmt(e.loc['Privada', 'media_geral'])}; "
              f"pública: {fmt(e.loc['Pública', 'media_geral'])} (diferença de {fmt(dif)} pontos)")
    print(f"  Renda '{pobre.renda_faixa}': média {fmt(pobre.media_geral)}; "
          f"'{rico.renda_faixa}': média {fmt(rico.media_geral)} (diferença de {fmt(rico.media_geral - pobre.media_geral)} pontos)")

In [ ]:
import matplotlib.pyplot as plt

AZUL, LARANJA, CINZA = "#2a78d6", "#eb6834", "#898781"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#c3c2b7", "axes.grid": True, "grid.color": "#e1e0d9",
                     "grid.linewidth": 0.8, "axes.axisbelow": True, "figure.dpi": 110})

def grafico_renda_escola(recorte, arquivo):
    d = ouro["renda_e_escola"].toPandas()
    d = d[d.recorte == recorte].sort_values("renda_grupo_ordem")
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for tipo, cor in (("Privada", LARANJA), ("Pública", AZUL)):
        s = d[d.tipo_escola == tipo]
        ax.plot(s.renda_grupo, s.media_geral, color=cor, linewidth=2, marker="o", markersize=6,
                markeredgecolor="white", markeredgewidth=1.5, label=f"Escola {tipo.lower()}")
        ultimo = s.iloc[-1]
        ax.annotate(f"{ultimo.media_geral:.0f}", (len(s) - 1, ultimo.media_geral), xytext=(8, 0),
                    textcoords="offset points", va="center", color="#52514e")
    ax.set_title(f"Média geral no ENEM 2023 por renda familiar ({recorte})", loc="left", fontsize=13, pad=12)
    ax.set_xlabel("Renda mensal da família (salários mínimos)")
    ax.set_ylabel("Média das 5 notas")
    ax.legend(frameon=False, loc="upper left")
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    fig.savefig(PASTA_GRAFICOS / arquivo, dpi=160)
    plt.show()

grafico_renda_escola("São Paulo", "1_renda_e_escola_sp.png")
grafico_renda_escola("Vale do Paraíba", "2_renda_e_escola_vale.png")

In [ ]:
# Municípios do Vale com pelo menos 30 concluintes em cada tipo de escola
m = municipios.dropna(subset=["media_publica", "media_privada"])
m = m[(m.concluintes_publica >= 30) & (m.concluintes_privada >= 30)].sort_values("media_publica")

fig, ax = plt.subplots(figsize=(9, 0.42 * len(m) + 1.4))
y = range(len(m))
ax.hlines(y, m.media_publica, m.media_privada, color="#d6d5cd", linewidth=2)
ax.scatter(m.media_publica, y, color=AZUL, s=48, zorder=3, edgecolor="white", linewidth=1.5, label="Escola pública")
ax.scatter(m.media_privada, y, color=LARANJA, s=48, zorder=3, edgecolor="white", linewidth=1.5, label="Escola privada")
for i, (_, linha) in enumerate(m.iterrows()):
    ax.annotate(f"+{linha.diferenca:.0f}", (linha.media_privada, i), xytext=(8, 0),
                textcoords="offset points", va="center", fontsize=10, color="#52514e")
ax.set_yticks(list(y), m.municipio)
ax.set_xlabel("Média das 5 notas (concluintes de 2023)")
ax.set_title("Distância entre escola pública e privada nos municípios do Vale do Paraíba", loc="left", fontsize=13, pad=30)
ax.legend(frameon=False, ncol=2, loc="lower left", bbox_to_anchor=(0, 1.0), borderaxespad=0.2)
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(PASTA_GRAFICOS / "3_municipios_vale.png", dpi=160)
plt.show()
m

In [ ]:
d = renda[renda.recorte == "Brasil"].dropna(subset=["renda_faixa"]).sort_values("renda_ordem")
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.plot(d.renda_faixa, d.media_geral, color=AZUL, linewidth=2, marker="o", markersize=6,
        markeredgecolor="white", markeredgewidth=1.5)
for _, linha in d.iloc[[0, -1]].iterrows():
    ax.annotate(f"{linha.media_geral:.0f}", (linha.renda_faixa, linha.media_geral), xytext=(0, 10),
                textcoords="offset points", ha="center", color="#52514e")
ax.set_title("Média geral no ENEM 2023 em cada faixa de renda familiar (Brasil)", loc="left", fontsize=13, pad=12)
ax.set_ylabel("Média das 5 notas")
ax.tick_params(axis="x", rotation=45)
for rotulo in ax.get_xticklabels():
    rotulo.set_horizontalalignment("right")
ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(PASTA_GRAFICOS / "4_renda_brasil.png", dpi=160)
plt.show()

## 8. Baixar os resultados

A célula abaixo junta as tabelas da camada ouro (em CSV, prontas para Excel ou Power BI) e os gráficos num zip, e inicia o download.

In [ ]:
arquivo = shutil.make_archive(str(BASE / "resultados_enem_2023"), "zip", PASTA_SAIDA)
print("Arquivo gerado:", arquivo)
try:
    from google.colab import files
    files.download(arquivo)
except ImportError:
    pass

## Para continuar estudando

- **Troque o recorte:** mude o filtro `uf_prova == "SP"` para o seu estado, ou troque a lista `VALE_DO_PARAIBA` pelos municípios da sua região.
- **Outras perguntas:** a prata já tem cor/raça, sexo e faixa etária. Tente uma tabela ouro de média geral por cor/raça dentro de cada faixa de renda.
- **Escala real:** o mesmo código roda sem mudanças no Databricks ou no AWS EMR. Basta trocar os caminhos locais por caminhos do S3, do ADLS ou do DBFS.

**Limitações:** o município considerado é o da prova, não o de residência. O tipo de escola só existe para os concluintes de 2023. A renda é declarada pelo próprio participante no questionário socioeconômico.

**Fonte:** Microdados do ENEM 2023, Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira (INEP).